#### 1. Imports & Setup

In [ ]:

import os
import re
import json
import pickle
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import fitz  
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer
import faiss

DATA_DIR = Path("../data")
ARTIFACT_DIR = Path("../data/artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("✅ Setup complete")

✅ Setup complete


#### 2. Data Loading

In [ ]:

df = pd.read_csv(DATA_DIR / "bank_customer_activity.csv")

def dataframe_to_text(df: pd.DataFrame) -> list[str]:
    """Convert each row into a readable sentence."""
    docs = []
    for _, row in df.iterrows():
        parts = [f"{col}: {val}" for col, val in row.items() if pd.notna(val)]
        docs.append("Customer Record — " + " | ".join(parts))
    return docs

customer_docs = dataframe_to_text(df)
print(f"✅ Loaded {len(customer_docs)} customer records from XLS")
print("Sample:", customer_docs[0][:200])

✅ Loaded 200000 customer records from XLS
Sample: Customer Record — Customer ID: C0000 | Customer Profile: Domestic Helper | Age: 19 | Gender: Male | Location: Bulawayo |  Monthly Income : 429 |  Average Savings Balance : 8,688.00 | Spending Habits: 


In [49]:
df

,Customer ID,Customer Profile,Age,Gender,Location,Monthly Income,Average Savings Balance,Spending Habits,Most Transactions,Loan History Performance,Credit Score,Last Viewed Products,Last Transaction,last_viewed_product_id,Most Recurring Transaction Type,Preferred Channel,Most Transaction Physical Location,Most Local/International Payment
0,C0000,Domestic Helper,19,Male,Bulawayo,429,"8,688.00",Heavy Spender,Evening,Average,C - Grade,Loan,Deposit,PLN001,Delay Repayments,Mobile USSD,Hotels/Lodges,Local
1,C0001,Young Entrepreneur,32,Female,Masvingo,137,"3,352.00",Heavy Spender,Afternoon,Good,D - Grade,Deposit,Withdrawal,DEP001,Late Payment Penalty,Mobile USSD,Hotels/Lodges,International
2,C0002,Farmer,65,Female,Bulawayo,102,"3,431.00",Low Spender,Evening,Average,A - Grade,Micro-Home Loan,Investment,PLN009,Interest Recalculation,Mobile Application,Hotels/Lodges,International
3,C0003,Pensioner,57,Female,Harare,352,"9,633.00",Moderate Spender,Night,Good,B - Grade,Deposit,Viewed Promotions,DEP001,Inter-Bank Transfers,POS,Online,International
4,C0004,Mature Entrepreneur,20,Female,Beitbridge,65,"2,212.00",Heavy Spender,Evening,Bad,D - Grade,Revolving Credit Line,Investment,PLN005,Balance Inquiry,In-Branch,Hotels/Lodges,Local
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,C199995,Mature Entrepreneur,43,Female,Harare,340,"6,955.00",Low Spender,Late Morning,Good,A - Grade,High-Yield Savings Account,Deposit,SAV001,Loan Application,Mobile USSD,Hotels/Lodges,International
199996,C199996,Professional White Collar,64,Female,Masvingo,300,"9,172.00",Low Spender,Night,Bad,A - Grade,Salary Overdraft,Loan Repayment,PLN007,Delay Repayments,POS,At Bank Premises,Local
199997,C199997,Professional White Collar,61,Male,Norton,203,"2,319.00",Moderate Spender,Late Morning,Good,A - Grade,Micro-Home Loan,Viewed Promotions,PLN009,Balance Inquiry,ATM,Hotels/Lodges,International
199998,C199998,Young Academic,44,Male,Harare,375,"9,794.00",Heavy Spender,Night,NaN,C - Grade,High-Yield Savings Account,Balance Inquiry,SAV001,Loan Application,ATM,Online,International


#### 3. Extracting text from PDFs (policy + catalogue)

In [ ]:

def extract_pdf_text(pdf_path: Path) -> list[str]:
    """Extract text page-by-page from a PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text("text").strip()
        if text:
            
            source_tag = f"[Source: {pdf_path.name} | Page {page_num + 1}]"
            pages.append(f"{source_tag}\n{text}")
    doc.close()
    print(f"  ↳ Extracted {len(pages)} pages from {pdf_path.name}")
    return pages

policy_docs    = extract_pdf_text(DATA_DIR / "bank_policy.pdf")
catalogue_docs = extract_pdf_text(DATA_DIR / "product_catalogue.pdf")

print(f"\n✅ PDF extraction complete")
print(f"   • Policy pages:    {len(policy_docs)}")
print(f"   • Catalogue pages: {len(catalogue_docs)}")

  ↳ Extracted 4 pages from bank_policy.pdf
  ↳ Extracted 37 pages from product_catalogue.pdf

✅ PDF extraction complete
   • Policy pages:    4
   • Catalogue pages: 37


#### 4. Chunking documents

In [ ]:

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 50] 

def chunk_documents(docs: list[str], source_label: str,
                    chunk_size=500, overlap=100) -> list[dict]:
    """Chunk a list of document strings and attach metadata."""
    chunked = []
    for doc in docs:
        for chunk in chunk_text(doc, chunk_size, overlap):
            chunked.append({
                "text":   chunk,
                "source": source_label
            })
    return chunked

customer_chunks  = chunk_documents(customer_docs,  "Customer Activity", chunk_size=400, overlap=80)
policy_chunks    = chunk_documents(policy_docs,    "Bank Policy",       chunk_size=500, overlap=100)
catalogue_chunks = chunk_documents(catalogue_docs, "Product Catalogue", chunk_size=500, overlap=100)

all_chunks = customer_chunks + policy_chunks + catalogue_chunks

print(f"✅ Total chunks: {len(all_chunks)}")
print(f"   • Customer:  {len(customer_chunks)}")
print(f"   • Policy:    {len(policy_chunks)}")
print(f"   • Catalogue: {len(catalogue_chunks)}")

✅ Total chunks: 400202
   • Customer:  400000
   • Policy:    19
   • Catalogue: 183


#### 5. Embedding chunks with Sentence Transformers

In [ ]:

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"   
embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in all_chunks]

print(f"Embedding {len(texts)} chunks … (may take a minute)")
embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"✅ Embeddings shape: {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 916.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 400202 chunks … (may take a minute)


Batches:  31%|███▏      | 1959/6254 [1:06:21<2:44:05,  2.29s/it]

#### 6. Building FAISS vector index

In [ ]:

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  
index.add(np.array(embeddings, dtype="float32"))

print(f"✅ FAISS index built — {index.ntotal} vectors, dim={dim}")

✅ FAISS index built — 400202 vectors, dim=384


#### 7. Saving artefacts so app.py can load them

In [ ]:

faiss.write_index(index, str(ARTIFACT_DIR / "bank_rag.index"))

with open(ARTIFACT_DIR / "chunks_metadata.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

config = {
    "embed_model": EMBED_MODEL_NAME,
    "total_chunks": len(all_chunks),
    "index_file": "bank_rag.index",
    "metadata_file": "chunks_metadata.pkl"
}
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Saved:")
print(f"   • {ARTIFACT_DIR}/bank_rag.index")
print(f"   • {ARTIFACT_DIR}/chunks_metadata.pkl")
print(f"   • {ARTIFACT_DIR}/config.json")

✅ Saved:
   • ..\data\artifacts/bank_rag.index
   • ..\data\artifacts/chunks_metadata.pkl
   • ..\data\artifacts/config.json


#### 8. Quick retrieval smoke-test

In [ ]:

def retrieve(query: str, top_k: int = 5) -> list[dict]:
    q_vec = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_vec, dtype="float32"), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = all_chunks[idx].copy()
        chunk["score"] = float(score)
        results.append(chunk)
    return results

test_q = "What are the loan eligibility requirements?"
hits = retrieve(test_q, top_k=3)
print(f"Query: {test_q}\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] Source: {h['source']} | Score: {h['score']:.4f}")
    print(f"    {h['text'][:200]}\n")

Query: What are the loan eligibility requirements?

[1] Source: Product Catalogue | Score: 0.6933
    n process; inquire about potential 
processing fees upfront to avoid surprises. 
Personalized Customer Support: Our dedicated team is here to guide you through the application 
process and answer any 

[2] Source: Bank Policy | Score: 0.6342
    [Source: bank_policy.pdf | Page 4]
5.1 Who May Borrow 
 
Eligibility to Borrow:  
o Individuals aged 18 years and above may apply for loans. 
o Minors may access borrowing options, provided a parent 

[3] Source: Product Catalogue | Score: 0.6221
    To qualify for the Investment Backed Loan, applicants must provide: 
Proof of Investments: Documentation to verify the value of the investments being used as 
collateral. 
Valid Identification: A gove



#### 9. RAG with Ollama

In [ ]:

import ollama

def build_context(hits: list[dict]) -> str:
    parts = []
    for i, h in enumerate(hits, 1):
        parts.append(f"[{i}] ({h['source']})\n{h['text']}")
    return "\n\n".join(parts)

def rag_answer(query: str, top_k: int = 5) -> str:
    hits    = retrieve(query, top_k)
    context = build_context(hits)

    prompt = f"""You are a helpful bank assistant. Use ONLY the context below to answer.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {query}
Answer:"""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

# ── Test ──
answer = rag_answer("What savings accounts does the bank offer?")
print(answer)

I don't have that information.
